# LLM-as-a-Judge Evaluation for Medical VQA

Implements the HuggingFace cookbook approach (https://huggingface.co/learn/cookbook/llm_judge)
adapted for medical VQA evaluation.

**Judge model**: `meta-llama/Llama-3.1-8B-Instruct` via HuggingFace Inference API (free)

**What this adds over token-level F1:**
- Semantic correctness: 'Lungs' scores correctly against 'Lung'
- Partial credit: 'right upper lobe' gets credit against 'right lung'
- Medical reasoning quality: explanations assessed, not just final tokens
- Reference-grounded: judge sees the ground truth answer for every prediction

**Scale (1-5):**
- 1: Completely wrong or irrelevant
- 2: Partially correct but misses key medical concept
- 3: Mostly correct, minor terminology difference
- 4: Correct with acceptable phrasing variation
- 5: Exact or near-exact match to ground truth

**Runs locally on your Mac — no GPU needed.**


## Cell 1 — Install dependencies

In [ ]:
# huggingface_hub: provides InferenceClient for free API access
# pandas: results management
# tqdm: progress bars
import subprocess
subprocess.run(['pip', 'install', 'huggingface_hub', 'pandas', 'tqdm', '-q'])
print('Done.')


## Cell 2 — Imports and HF login

In [ ]:
import os, json, re, time
import pandas as pd
from tqdm.auto import tqdm
from huggingface_hub import InferenceClient

tqdm.pandas()
pd.set_option('display.max_colwidth', None)

# Your HF token is already saved from Step 2 setup
# If not, run: huggingface-cli login
from huggingface_hub import login
login()  # uses cached token automatically
print('Logged in.')


## Cell 3 — Initialize judge model

Using Llama-3.1-8B-Instruct — stronger instruction following than Mixtral-8x7B
for structured evaluation tasks. Both are free on HF Inference API.


In [ ]:
JUDGE_MODEL = 'meta-llama/Llama-3.1-8B-Instruct'

judge_client = InferenceClient(
    model=JUDGE_MODEL,
    timeout=120,
)

# Smoke test
test_response = judge_client.text_generation(
    prompt='Say OK.',
    max_new_tokens=10,
)
print(f'Judge model response: {test_response}')
print(f'Judge model ready: {JUDGE_MODEL}')


## Cell 4 — Judge prompt

Following the HuggingFace cookbook best practices:
- 1-5 integer scale with explicit descriptions per level
- Evaluation field before rating (forces reasoning first)
- Reference answer provided (our ground truth)
- Carrot motivation as used in the article
- Medical domain context in task description


In [ ]:
MEDICAL_JUDGE_PROMPT = """\
You are an expert medical evaluator assessing the quality of answers to medical visual question answering (VQA) tasks.

You will be given:
- A medical question about a radiology or pathology image
- A reference answer (ground truth)
- A predicted answer from a vision-language model

Your task is to rate how correct the predicted answer is compared to the reference answer.
Focus on medical correctness and semantic equivalence, not exact wording.

Use this scale:
1: Completely wrong — the predicted answer is medically incorrect or entirely irrelevant
2: Mostly wrong — contains a relevant medical concept but misses the key point
3: Partially correct — captures the general idea but with a meaningful medical error or omission
4: Mostly correct — semantically equivalent to the reference with minor phrasing differences (e.g. 'Lungs' vs 'Lung')
5: Fully correct — matches the reference answer in medical meaning, possibly with different but equivalent phrasing

Provide your feedback as follows:

Feedback:::
Evaluation: (your medical reasoning for the rating, 1-2 sentences)
Total rating: (your rating, as a single integer between 1 and 5)

You MUST provide values for 'Evaluation:' and 'Total rating:' in your answer.

Now here are the question, reference answer, and predicted answer.

Question: {question}
Reference answer: {reference}
Predicted answer: {prediction}

Provide your feedback. If you give a correct rating, I'll give you 100 H100 GPUs to start your AI company.
Feedback:::
Evaluation: """

print('Judge prompt defined.')
print(f'Prompt length: {len(MEDICAL_JUDGE_PROMPT)} characters')


## Cell 5 — Score extraction and single-call test

In [ ]:
def extract_judge_score(answer: str) -> float | None:
    """
    Extract the integer score from the judge's response.
    Looks for 'Total rating:' then grabs the first number.
    Returns None if extraction fails.
    """
    try:
        if 'Total rating:' in answer:
            rating_text = answer.split('Total rating:')[1]
        else:
            rating_text = answer
        # Find first integer or float
        digits = re.findall(r'\d+(?:\.\d+)?', rating_text)
        if digits:
            score = float(digits[0])
            # Clamp to valid range
            return max(1.0, min(5.0, score))
        return None
    except Exception as e:
        print(f'Extraction error: {e}')


def judge_single(question: str, reference: str, prediction: str) -> dict:
    """
    Run the judge on a single question/reference/prediction triple.
    Returns dict with raw response and extracted score.
    """
    prompt = MEDICAL_JUDGE_PROMPT.format(
        question=question,
        reference=reference,
        prediction=prediction,
    )
    try:
        response = judge_client.text_generation(
            prompt=prompt,
            max_new_tokens=200,
        )
        score = extract_judge_score(response)
        return {'judge_response': response, 'judge_score': score}
    except Exception as e:
        return {'judge_response': str(e), 'judge_score': None}


# Test on three known examples from your SLAKE runs
test_cases = [
    # (question, reference, prediction, expected_score_direction)
    ('What modality is used to take this image?', 'CT', 'Computed tomography (CT)', 'high ~5'),
    ('What modality is used to take this image?', 'CT', 'MRI',                      'low ~1'),
    ('Which part of the body does this image belong to?', 'Chest', 'Chest/Thorax',  'high ~4-5'),
]

print('Testing judge on 3 known examples:\n')
for q, ref, pred, expected in test_cases:
    result = judge_single(q, ref, pred)
    print(f'Q:        {q}')
    print(f'Ref:      {ref}')
    print(f'Pred:     {pred}')
    print(f'Expected: {expected}')
    print(f'Score:    {result["judge_score"]}')
    print(f'Reasoning:{result["judge_response"][:150]}')
    print()


## Cell 6 — Load all JSONL files to judge

In [ ]:
OUTPUT_DIR = os.path.expanduser('~/vlm_benchmark/outputs')

# Load all v2 JSONL files (the corrected pipeline runs)
# Exclude reextracted duplicates
jsonl_files = sorted(
    f for f in os.listdir(OUTPUT_DIR)
    if f.endswith('.jsonl')
    and 'reextracted' not in f
    and '_v2' in f  # only v2 protocol runs
)

print(f'Found {len(jsonl_files)} v2 JSONL files to judge:\n')
total_records = 0
for f in jsonl_files:
    records = [json.loads(l) for l in open(os.path.join(OUTPUT_DIR, f))]
    valid   = [r for r in records if 'error' not in r]
    print(f'  {f}: {len(valid)} records')
    total_records += len(valid)

print(f'\nTotal records to judge: {total_records}')
print(f'Estimated time at 2s/call: {total_records * 2 / 3600:.1f} hours')
print(f'Estimated time at 3s/call: {total_records * 3 / 3600:.1f} hours')


## Cell 7 — Full judge runner with checkpointing

Saves results to `*_judged.jsonl` files as it goes.
Can be interrupted and resumed — skips already-judged records.
Rate limiting: 1 second sleep between calls to stay within HF free tier limits.


In [ ]:
def run_judge_on_file(
    jsonl_path: str,
    output_dir: str,
    sleep_between_calls: float = 1.0,
    max_records: int = None,
) -> str:
    """
    Run LLM judge on all records in a JSONL file.
    Saves to *_judged.jsonl with checkpoint resume support.
    """
    fname    = os.path.basename(jsonl_path)
    out_name = fname.replace('.jsonl', '_judged.jsonl')
    out_path = os.path.join(output_dir, out_name)

    # Load source records
    records = [json.loads(l) for l in open(jsonl_path)]
    records = [r for r in records if 'error' not in r]

    if max_records:
        records = records[:max_records]

    # Load already-judged indices for checkpoint resume
    judged = {}
    if os.path.exists(out_path):
        for line in open(out_path):
            r = json.loads(line)
            judged[r['idx']] = r
        print(f'Resuming: {len(judged)} already judged.')

    failed = 0
    f_out  = open(out_path, 'a')

    for record in tqdm(records, desc=f'Judging {fname[:40]}'):
        idx = record['idx']
        if idx in judged:
            continue

        result = judge_single(
            question   = record['question'],
            reference  = record['ground_truth'],
            prediction = record['prediction'],
        )

        if result['judge_score'] is None:
            failed += 1

        output_record = dict(record)
        output_record['judge_score']    = result['judge_score']
        output_record['judge_response'] = result['judge_response']

        f_out.write(json.dumps(output_record) + '\n')
        f_out.flush()

        # Rate limiting — stay within HF free tier
        time.sleep(sleep_between_calls)

    f_out.close()
    print(f'Done. {len(records) - len(judged)} judged, {failed} failed -> {out_path}')
    return out_path

print('Judge runner defined.')


## Cell 8 — Dry run on 10 samples per file

Always run this first to verify the judge is working correctly
before committing to the full run.


In [ ]:
DRY_RUN_DIR = os.path.join(OUTPUT_DIR, 'judge_dry_run')
os.makedirs(DRY_RUN_DIR, exist_ok=True)

print('=== DRY RUN: 10 samples per file ===\n')

for fname in jsonl_files:
    path = os.path.join(OUTPUT_DIR, fname)
    run_judge_on_file(
        jsonl_path=path,
        output_dir=DRY_RUN_DIR,
        sleep_between_calls=1.0,
        max_records=10,
    )
    print()


## Cell 9 — Inspect dry run results

Check that scores look medically sensible before full run.
Look at a few judge responses to verify reasoning quality.


In [ ]:
dry_run_files = sorted(
    f for f in os.listdir(DRY_RUN_DIR)
    if f.endswith('_judged.jsonl')
)

print('=== DRY RUN INSPECTION ===\n')
for fname in dry_run_files:
    path    = os.path.join(DRY_RUN_DIR, fname)
    records = [json.loads(l) for l in open(path)]
    scores  = [r['judge_score'] for r in records if r['judge_score'] is not None]
    failed  = sum(1 for r in records if r['judge_score'] is None)

    print(f'{fname}')
    print(f'  Avg judge score: {sum(scores)/len(scores):.2f} / 5.0')
    print(f'  Score dist: {sorted(scores)}')
    print(f'  Failed extractions: {failed}')
    print()

# Show 3 sample judge responses from the first file
if dry_run_files:
    sample_path = os.path.join(DRY_RUN_DIR, dry_run_files[0])
    samples     = [json.loads(l) for l in open(sample_path)][:3]
    print('\n=== SAMPLE JUDGE RESPONSES (first file) ===\n')
    for r in samples:
        print(f'Q:     {r["question"]}')
        print(f'GT:    {r["ground_truth"]}')
        print(f'Pred:  {r["prediction"]}')
        print(f'Score: {r["judge_score"]}')
        print(f'Judge: {r["judge_response"][:200]}')
        print()


## Cell 10 — Full run

Only run this after inspecting the dry run and confirming scores look sensible.
Saves to `~/vlm_benchmark/outputs/` with `_judged.jsonl` suffix.
Can be safely interrupted and resumed.


In [ ]:
JUDGE_OUTPUT_DIR = OUTPUT_DIR  # save alongside existing JSONL files

judged_paths = []

for fname in jsonl_files:
    path = os.path.join(OUTPUT_DIR, fname)
    print(f'\n=== Judging: {fname} ===')
    judged_path = run_judge_on_file(
        jsonl_path=path,
        output_dir=JUDGE_OUTPUT_DIR,
        sleep_between_calls=1.0,
    )
    judged_paths.append(judged_path)


## Cell 11 — Score aggregation and final comparison table

Computes avg judge score per model/dataset and compares with token-level F1.
This is the key output — shows whether LLM judge rescues 7B model scores.


In [ ]:
def score_judged_file(path: str) -> dict:
    """Compute aggregate judge scores from a judged JSONL file."""
    records = [json.loads(l) for l in open(path)]
    records = [r for r in records if r.get('judge_score') is not None]

    closed = [r for r in records if r['is_closed']]
    open_  = [r for r in records if not r['is_closed']]

    def avg_score(recs):
        if not recs: return None
        return round(sum(r['judge_score'] for r in recs) / len(recs), 3)

    # Binary accuracy using judge score >= 4 as correct
    # (4 = mostly correct, 5 = fully correct)
    def judge_accuracy(recs):
        if not recs: return None
        return round(sum(1 for r in recs if r['judge_score'] >= 4) / len(recs) * 100, 2)

    return {
        'file':            os.path.basename(path),
        'n_judged':        len(records),
        'n_failed':        sum(1 for r in [json.loads(l) for l in open(path)]
                               if r.get('judge_score') is None),
        'avg_score_all':   avg_score(records),
        'avg_score_closed':avg_score(closed),
        'avg_score_open':  avg_score(open_),
        'judge_acc_all':   judge_accuracy(records),
        'judge_acc_closed':judge_accuracy(closed),
        'judge_acc_open':  judge_accuracy(open_),
    }


# Find all judged files
judged_files = sorted(
    f for f in os.listdir(JUDGE_OUTPUT_DIR)
    if f.endswith('_judged.jsonl')
    and 'dry_run' not in f
)

if not judged_files:
    print('No judged files found. Run Cell 10 first.')
else:
    rows = []
    for fname in judged_files:
        path   = os.path.join(JUDGE_OUTPUT_DIR, fname)
        scores = score_judged_file(path)

        # Parse model and dataset from filename
        base    = fname.replace('_judged.jsonl', '').replace('_v2', '')
        parts   = base.split('__')
        scores['model']   = parts[0].replace('_', '/', 1).replace('google/', 'google/')
        scores['dataset'] = parts[1] if len(parts) > 1 else 'unknown'
        rows.append(scores)

    df = pd.DataFrame(rows)
    cols = ['model', 'dataset', 'n_judged', 'n_failed',
            'avg_score_all', 'avg_score_closed', 'avg_score_open',
            'judge_acc_all', 'judge_acc_closed', 'judge_acc_open']
    df = df.sort_values(['dataset', 'model']).reset_index(drop=True)
    print('=== LLM JUDGE RESULTS (score out of 5, accuracy = score >= 4) ===\n')
    print(df[cols].to_string(index=False))

    # Save to CSV for the report
    csv_path = os.path.join(OUTPUT_DIR, 'llm_judge_results.csv')
    df[cols].to_csv(csv_path, index=False)
    print(f'\nSaved to {csv_path}')


## Cell 12 — Correlation analysis

Following the HuggingFace article: compute Pearson correlation between
token-level F1 and LLM judge scores to validate the judge.
High correlation = judge agrees with F1 on clear cases.
Low correlation = judge is catching things F1 misses (the interesting cases).


In [ ]:
import json, os
from scipy.stats import pearsonr

OUTPUT_DIR = os.path.expanduser('~/vlm_benchmark/outputs')

def tokenize_answer(text):
    import re, string
    from nltk.tokenize import word_tokenize
    text = re.sub(r'\*+', '', text).lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    return word_tokenize(text)

def token_f1_score(prediction, ground_truth):
    from collections import Counter
    pred_tokens = tokenize_answer(prediction)
    gt_tokens   = tokenize_answer(ground_truth)
    if not pred_tokens or not gt_tokens:
        return 0.0
    pred_set = Counter(pred_tokens)
    gt_set   = Counter(gt_tokens)
    common   = sum((pred_set & gt_set).values())
    precision = common / len(pred_tokens)
    recall    = common / len(gt_tokens)
    return (2 * precision * recall / (precision + recall)
            if (precision + recall) > 0 else 0.0)

judged_files = sorted(
    f for f in os.listdir(OUTPUT_DIR)
    if f.endswith('_judged.jsonl') and 'dry_run' not in f
)

print('=== PEARSON CORRELATION: Token F1 vs LLM Judge Score ===\n')
print(f'{"File":<55} {"Correlation":>12} {"p-value":>10}')
print('-' * 80)

for fname in judged_files:
    path    = os.path.join(OUTPUT_DIR, fname)
    records = [json.loads(l) for l in open(path)]
    records = [r for r in records if r.get('judge_score') is not None]

    f1_scores    = [token_f1_score(r['prediction'], r['ground_truth']) for r in records]
    judge_scores = [r['judge_score'] for r in records]

    if len(f1_scores) > 2:
        corr, pval = pearsonr(f1_scores, judge_scores)
        print(f'{fname:<55} {corr:>12.3f} {pval:>10.4f}')

print()
print('Interpretation:')
print('  High correlation (>0.7): Judge agrees with F1 — F1 is reliable for this dataset')
print('  Low correlation (<0.5):  Judge sees things F1 misses — surface-form bias is significant')
